# RIPE and PeeringDB stats

In [ ]:
import os
import json
import pickle
import datetime
import networkx as nx
import pandas as pd
import numpy as np
from multiprocessing import Pool
import matplotlib.pyplot as plt
from tqdm import tqdm
from matplotlib.ticker import FixedLocator, FixedFormatter


In [ ]:
font_size = 24
scale_factor = 1.2
two_sided_font_size = font_size * scale_factor

plt.rcParams["font.size"] = font_size


ipv_color = {4: "tab:blue", 6: "tab:green"}


In [ ]:
REPO_ROOT = os.path.abspath("..")
fd = open(os.path.join(REPO_ROOT, "settings.json"))
parameters = json.load(fd)
for _k in ("DATA_DIR", "DATA_RAW_DIR", "IMAGE_DIR", "WORKING_DIR", "VISIBILITY_OUTPUT_DIR", "VISIBILITY_ANNOUNCED_OUTPUT_DIR"):
    if isinstance(parameters.get(_k), str) and not os.path.isabs(parameters[_k]):
        parameters[_k] = os.path.normpath(os.path.join(REPO_ROOT, parameters[_k]))
fd.close()

data_dir = parameters["DATA_DIR"]
data_raw_dir = parameters["DATA_RAW_DIR"]
start_date = parameters["START_DATE"]
end_date = parameters["END_DATE"]
collectors = parameters["COLLECTORS"]
image_dir = parameters["IMAGE_DIR"]

# start_date = datetime.datetime.strptime(start_date, "%Y-%m-%d")
# end_date = datetime.datetime.strptime(end_date, "%Y-%m-%d")


In [ ]:
## Open stats output file (overwrites on every run)
numbers_dir = f"{data_dir}/processed/numbers"
os.makedirs(numbers_dir, exist_ok=True)
_stats = open(f"{numbers_dir}/8-RIPE_PeeringDB_stats.md", "w")
_stats.write("# Stats: 8-RIPE_PeeringDB_stats\n\n")
_stats.write(f"*Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}*\n\n")
print("Stats file opened.")


## Load data

### Historical PeeringDB data

In [ ]:
selected_dates = [
    datetime.datetime(2022, 1, 1),
    datetime.datetime(2022, 7, 1),
    datetime.datetime(2023, 1, 1),
    datetime.datetime(2023, 7, 1),
    datetime.datetime(2024, 1, 1),
    datetime.datetime(2024, 7, 1),
    datetime.datetime(2025, 1, 1),
    datetime.datetime(2025, 7, 1),
    datetime.datetime(2026, 1, 1),
]

historical_last_date = selected_dates[-1]

historical_df = {}

for date in tqdm(selected_dates):
    filename = f"{data_raw_dir}/peeringdb/peeringdb_2_dump_{date.year}_{date.month:02d}_{date.day:02d}.json"
    fd = open(filename, "r")
    data = json.load(fd)
    fd.close()

    df = pd.DataFrame(data["net"]["data"])
    df["updated"] = df["updated"].apply(
        lambda updated: datetime.datetime.strptime(updated, "%Y-%m-%dT%H:%M:%SZ")
    )
    historical_df[date] = df.copy()


### PeeringDB data

In [ ]:
filename = (
    f"{data_dir}/processed/peeringdb/prefix_limit_peeringdb_{start_date}_{end_date}.pkl"
)
df_peeringdb = pd.read_pickle(filename)
df_peeringdb.head(2)


### Prefix Origin AS


In [ ]:
filename = f"{data_dir}/processed/timeseries_prefix_originated_visibility.pkl"

with open(filename, "rb") as fd:
    originated_prefixes = pickle.load(fd)


## RIS and PeeringDB

### RIS

In [ ]:
asns_ris = set(originated_prefixes.keys())
asns_ris_exclusively_ipv4 = set()
asns_ris_exclusively_ipv6 = set()
asns_ris_both = set()

for asn in asns_ris:
    asn_data = originated_prefixes[asn]
    ipv4_visibility = len(asn_data[4]["total"])
    ipv6_visibility = len(asn_data[6]["total"])

    if ipv4_visibility > 0 and ipv6_visibility == 0:
        asns_ris_exclusively_ipv4.add(asn)
    elif ipv4_visibility == 0 and ipv6_visibility > 0:
        asns_ris_exclusively_ipv6.add(asn)
    elif ipv4_visibility > 0 and ipv6_visibility > 0:
        asns_ris_both.add(asn)
    else:
        print(f"ASN {asn} has no prefixes in either IPv4 or IPv6.")

n_asns_ris = len(asns_ris)
n_asns_ris_exclusively_ipv4 = len(asns_ris_exclusively_ipv4)
n_asns_ris_exclusively_ipv6 = len(asns_ris_exclusively_ipv6)
n_asns_ris_both = len(asns_ris_both)

print(f"ASes in RIS: {n_asns_ris}")
print(f"ASes with exclusively IPv4 prefixes: {n_asns_ris_exclusively_ipv4}")
print(f"ASes with exclusively IPv6 prefixes: {n_asns_ris_exclusively_ipv6}")
print(f"ASes with both IPv4 and IPv6 prefixes: {n_asns_ris_both}")

assert (
    n_asns_ris
    == n_asns_ris_exclusively_ipv4 + n_asns_ris_exclusively_ipv6 + n_asns_ris_both
)


In [ ]:
_stats.write("## RIPE RIS\n\n")
_stats.write(f"- ASes in RIS: {n_asns_ris:,}\n")
_stats.write(f"  - IPv4 only: {n_asns_ris_exclusively_ipv4:,}\n")
_stats.write(f"  - IPv6 only: {n_asns_ris_exclusively_ipv6:,}\n")
_stats.write(f"  - Both IPv4 and IPv6: {n_asns_ris_both:,}\n")
_stats.write("\n")


### PeeringDB

In [ ]:
asns_peeringdb = set(df_peeringdb["asn"])
print(f"ASes in PeeringDB: {len(asns_peeringdb)}")


#### Recent PeeringDB stats

In [ ]:
year_recent = 2023
year_recent_datetime = datetime.datetime(year_recent, 1, 1)
df_peeringdb_recent = df_peeringdb[df_peeringdb["updated"].dt.year >= year_recent]
asns_peeringdb_recent = set(df_peeringdb_recent["asn"])

print(
    f"ASes in PeeringDB snapshot which their update date is recent than {year_recent} (included): {len(asns_peeringdb_recent)}"
)


In [ ]:
_stats.write("## PeeringDB\n\n")
_stats.write(f"- Total entries: {len(asns_peeringdb):,}\n")
_stats.write(
    f"- Entries updated >= {year_recent}: {len(asns_peeringdb_recent):,}"
    f" ({len(asns_peeringdb_recent)/len(asns_peeringdb)*100:.1f}%)\n"
)
_stats.write("\n")


## How many entries in PeeringDB have zero limits?

In [ ]:
n_entries = len(df_peeringdb_recent)

valid_entries_ipv4 = df_peeringdb_recent["limits_ipv4"].notna()
valid_entries_ipv6 = df_peeringdb_recent["limits_ipv6"].notna()
valid_entries_ipv4_and_ipv6 = valid_entries_ipv4 & valid_entries_ipv6

invalid_entries_ipv4 = df_peeringdb_recent["limits_ipv4"].isna()
invalid_entries_ipv6 = df_peeringdb_recent["limits_ipv6"].isna()
invalid_entries_ipv4_or_ipv6 = invalid_entries_ipv4 | invalid_entries_ipv6
invalid_entries_ipv4_and_ipv6 = invalid_entries_ipv4 & invalid_entries_ipv6


print(f"PeeringDB 2025-01-01 to 2025-12-31, entries: {n_entries}")
print(f"Valid entries (IPv4): {valid_entries_ipv4.sum()} ({valid_entries_ipv4.mean() * 100:.2f}%)")
print(f"Valid entries (IPv6): {valid_entries_ipv6.sum()} ({valid_entries_ipv6.mean() * 100:.2f}%)")
print(f"Valid entries (IPv4 and IPv6): {valid_entries_ipv4_and_ipv6.sum()} ({valid_entries_ipv4_and_ipv6.mean() * 100:.2f}%)")
print(f"Invalid entries (IPv4): {invalid_entries_ipv4.sum()} ({invalid_entries_ipv4.mean() * 100:.2f}%)")
print(f"Invalid entries (IPv6): {invalid_entries_ipv6.sum()} ({invalid_entries_ipv6.mean() * 100:.2f}%)")
print(f"Invalid entries (IPv4 or IPv6): {invalid_entries_ipv4_or_ipv6.sum()} ({invalid_entries_ipv4_or_ipv6.mean() * 100:.2f}%)")
print(f"Invalid entries (IPv4 and IPv6): {invalid_entries_ipv4_and_ipv6.sum()} ({invalid_entries_ipv4_and_ipv6.mean() * 100:.2f}%)")

In [ ]:
_stats.write("## Prefix Limits (recent PeeringDB entries)\n\n")
_stats.write(f"- Total recent entries: {n_entries:,}\n")
_stats.write(f"- Valid IPv4 limits: {valid_entries_ipv4.sum():,} ({valid_entries_ipv4.mean()*100:.2f}%)\n")
_stats.write(f"- Valid IPv6 limits: {valid_entries_ipv6.sum():,} ({valid_entries_ipv6.mean()*100:.2f}%)\n")
_stats.write(f"- Zero/null IPv4 limits: {invalid_entries_ipv4.sum():,} ({invalid_entries_ipv4.mean()*100:.2f}%)\n")
_stats.write(f"- Zero/null IPv6 limits: {invalid_entries_ipv6.sum():,} ({invalid_entries_ipv6.mean()*100:.2f}%)\n")
_stats.write(f"- Zero/null either (IPv4 or IPv6): {invalid_entries_ipv4_or_ipv6.sum():,} ({invalid_entries_ipv4_or_ipv6.mean()*100:.2f}%)\n")
_stats.write(f"- Zero/null both (IPv4 and IPv6): {invalid_entries_ipv4_and_ipv6.sum():,} ({invalid_entries_ipv4_and_ipv6.mean()*100:.2f}%)\n")
_stats.write("\n")

### Intersection of RIS and PeeringDB

In [ ]:
# Intersection of RIPE RIS and PeeringDB ASNs
asns_ris_peeringdb = asns_ris & asns_peeringdb

asns_ris_peeringdb_exclusively_ipv4 = asns_ris_exclusively_ipv4 & asns_peeringdb
asns_ris_peeringdb_exclusively_ipv6 = asns_ris_exclusively_ipv6 & asns_peeringdb
asns_ris_peeringdb_both = asns_ris_both & asns_peeringdb

n_asns_ris_peeringdb = len(asns_ris_peeringdb)
n_asns_ris_peeringdb_exclusively_ipv4 = len(asns_ris_peeringdb_exclusively_ipv4)
n_asns_ris_peeringdb_exclusively_ipv6 = len(asns_ris_peeringdb_exclusively_ipv6)
n_asns_ris_peeringdb_both = len(asns_ris_peeringdb_both)

# Recent - Intersection of RIPE RIS and PeeringDB ASes
asns_ris_peeringdb_recent = asns_ris_peeringdb & asns_peeringdb_recent
asns_ris_peeringdb_exclusively_ipv4_recent = (
    asns_ris_peeringdb_exclusively_ipv4 & asns_peeringdb_recent
)
asns_ris_peeringdb_exclusively_ipv6_recent = (
    asns_ris_peeringdb_exclusively_ipv6 & asns_peeringdb_recent
)
asns_ris_peeringdb_both_recent = asns_ris_peeringdb_both & asns_peeringdb_recent

n_asns_ris_peeringdb_recent = len(asns_ris_peeringdb_recent)
n_asns_ris_peeringdb_exclusively_ipv4_recent = len(
    asns_ris_peeringdb_exclusively_ipv4_recent
)
n_asns_ris_peeringdb_exclusively_ipv6_recent = len(
    asns_ris_peeringdb_exclusively_ipv6_recent
)
n_asns_ris_peeringdb_both_recent = len(asns_ris_peeringdb_both_recent)


In [ ]:
_stats.write("## Working Dataset (RIS ∩ PeeringDB recent)\n\n")
_stats.write(f"- RIS ∩ PeeringDB (any): {n_asns_ris_peeringdb:,}\n")
_stats.write(
    f"- RIS ∩ PeeringDB (updated >= {year_recent}): {n_asns_ris_peeringdb_recent:,}"
    f" ({n_asns_ris_peeringdb_recent/n_asns_ris*100:.2f}% of RIS)\n"
)
_stats.write(f"  - IPv4 only: {n_asns_ris_peeringdb_exclusively_ipv4_recent:,}\n")
_stats.write(f"  - IPv6 only: {n_asns_ris_peeringdb_exclusively_ipv6_recent:,}\n")
_stats.write(f"  - Both: {n_asns_ris_peeringdb_both_recent:,}\n")
_stats.write("\n")


In [ ]:
# Non-zero filter: ASes in working dataset with at least one non-zero limit
df_ris_peeringdb_recent_filtered = df_peeringdb_recent[
    df_peeringdb_recent["asn"].isin(asns_ris_peeringdb_recent)
].copy()

valid_ris_ipv4 = df_ris_peeringdb_recent_filtered["limits_ipv4"].notna()
valid_ris_ipv6 = df_ris_peeringdb_recent_filtered["limits_ipv6"].notna()
valid_ris_either = valid_ris_ipv4 | valid_ris_ipv6

asns_ris_peeringdb_recent_nonzero = set(
    df_ris_peeringdb_recent_filtered[valid_ris_either]["asn"]
)

n_asns_ris_peeringdb_recent_nonzero = len(asns_ris_peeringdb_recent_nonzero)
n_asns_ris_peeringdb_recent_nonzero_excl_ipv4 = len(
    asns_ris_peeringdb_exclusively_ipv4_recent & asns_ris_peeringdb_recent_nonzero
)
n_asns_ris_peeringdb_recent_nonzero_excl_ipv6 = len(
    asns_ris_peeringdb_exclusively_ipv6_recent & asns_ris_peeringdb_recent_nonzero
)
n_asns_ris_peeringdb_recent_nonzero_both = len(
    asns_ris_peeringdb_both_recent & asns_ris_peeringdb_recent_nonzero
)

print(f"ASes in working dataset with at least one non-zero limit: {n_asns_ris_peeringdb_recent_nonzero}")
print(f"  - IPv4 only: {n_asns_ris_peeringdb_recent_nonzero_excl_ipv4}")
print(f"  - IPv6 only: {n_asns_ris_peeringdb_recent_nonzero_excl_ipv6}")
print(f"  - Both: {n_asns_ris_peeringdb_recent_nonzero_both}")

_stats.write("## Working Dataset (non-zero limits)\n\n")
_stats.write(
    f"- RIS ∩ PeeringDB (updated >= {year_recent}, non-zero): {n_asns_ris_peeringdb_recent_nonzero:,}"
    f" ({n_asns_ris_peeringdb_recent_nonzero/n_asns_ris*100:.2f}% of RIS)\n"
)
_stats.write(f"  - IPv4 only: {n_asns_ris_peeringdb_recent_nonzero_excl_ipv4:,}\n")
_stats.write(f"  - IPv6 only: {n_asns_ris_peeringdb_recent_nonzero_excl_ipv6:,}\n")
_stats.write(f"  - Both: {n_asns_ris_peeringdb_recent_nonzero_both:,}\n")
_stats.write("\n")


In [ ]:
(
    n_asns_ris_peeringdb_exclusively_ipv4
    + n_asns_ris_peeringdb_exclusively_ipv6
    + n_asns_ris_peeringdb_both
)


### Plot

In [ ]:
data_plot = {
    "All RIS\nASes": {
        "total": n_asns_ris,
        "exclusive_ipv4": n_asns_ris_exclusively_ipv4,
        "exclusive_ipv6": n_asns_ris_exclusively_ipv6,
        "both": n_asns_ris_both,
    },
    "∩ PeeringDB": {
        "total": n_asns_ris_peeringdb,
        "fraction": n_asns_ris_peeringdb / n_asns_ris,
        "exclusive_ipv4": n_asns_ris_peeringdb_exclusively_ipv4,
        "exclusive_ipv6": n_asns_ris_peeringdb_exclusively_ipv6,
        "both": n_asns_ris_peeringdb_both,
    },
    "∩ Updated\n≥ 2023": {
        "total": n_asns_ris_peeringdb_recent,
        "fraction": n_asns_ris_peeringdb_recent / n_asns_ris,
        "exclusive_ipv4": n_asns_ris_peeringdb_exclusively_ipv4_recent,
        "exclusive_ipv6": n_asns_ris_peeringdb_exclusively_ipv6_recent,
        "both": n_asns_ris_peeringdb_both_recent,
    },
    "∩ Non-zero\nlimit": {
        "total": n_asns_ris_peeringdb_recent_nonzero,
        "fraction": n_asns_ris_peeringdb_recent_nonzero / n_asns_ris,
        "exclusive_ipv4": n_asns_ris_peeringdb_recent_nonzero_excl_ipv4,
        "exclusive_ipv6": n_asns_ris_peeringdb_recent_nonzero_excl_ipv6,
        "both": n_asns_ris_peeringdb_recent_nonzero_both,
    },
}


In [ ]:
df_peeringdb.head(2)


In [ ]:
plt.figure(figsize=(15, 8))

for index, (category, stats) in enumerate(data_plot.items()):
    plt.bar(
        index,
        stats["exclusive_ipv4"],
        color="tab:blue",
        alpha=0.9,
    )
    plt.bar(
        index,
        stats["exclusive_ipv6"],
        bottom=stats["exclusive_ipv4"],
        color="tab:green",
        alpha=0.9,
    )
    plt.bar(
        index,
        stats["both"],
        bottom=stats["exclusive_ipv4"] + stats["exclusive_ipv6"],
        color="teal",
        alpha=0.5,
    )

    total = stats["total"]
    if index == 0:
        plt.text(
            index, total * 1.03, f"{total}", ha="center", va="bottom", color="black"
        )
    else:
        plt.text(
            index,
            total * 1.03,
            f"{total}\n({stats['fraction']:.1%})",
            ha="center",
            va="bottom",
            color="black",
        )


plt.xticks([0, 1, 2, 3], list(data_plot.keys()))
plt.xlim(-0.5, 3.5)
plt.ylabel("Number of ASes")
plt.ylim(0, n_asns_ris * 1.2)

plt.bar([0], [0], color="tab:blue", alpha=0.7, label="Exclusive IPv4")
plt.bar([0], [0], color="tab:green", alpha=0.7, label="Exclusive IPv6")
plt.bar([0], [0], color="teal", alpha=0.5, label="Both IPv4 and IPv6")

plt.legend(loc="upper right", bbox_to_anchor=(1, 1.15), ncol=3, frameon=False, fontsize=1.09*font_size)

plt.savefig(
    f"{image_dir}/peering_db/ripe_peeringdb_asn_stats.pdf", bbox_inches="tight", dpi=300
)
plt.savefig(
    f"{image_dir}/peering_db/ripe_peeringdb_asn_stats.png", bbox_inches="tight", dpi=300
)
plt.show()


In [ ]:
df_ris_peeringdb_recent = df_peeringdb[
    df_peeringdb["asn"].isin(asns_ris_peeringdb_recent_nonzero)
].copy()

df_ris_peeringdb_recent["info_type"] = df_ris_peeringdb_recent["info_type"].apply(
    lambda info_type: info_type if info_type != "" else "N/A"
)

asn_info_type = df_ris_peeringdb_recent["info_type"].value_counts().to_dict()


In [ ]:
def asn_info_type_to_latex(
    asn_info_type: dict, date: str, label: str = "tab:coverage-by-type"
) -> str:
    total = sum(asn_info_type.values())
    rows = []
    for network_type, count in asn_info_type.items():
        pct = count / total * 100
        rows.append(f"    {network_type} & {count:,} & {pct:.1f}\\% \\\\")

    rows_str = "\n".join(rows)
    return rf"""\begin{{table}}[t]
\centering
\caption{{PeeringDB coverage by network type as {date}}}
\label{{{label}}}
\begin{{tabular}}{{@{{}}lrr@{{}}}}
\toprule
\textbf{{Network Type}} & \textbf{{Visible ASes}} & \textbf{{Percentage}} \\
\midrule
{rows_str}
\bottomrule
\end{{tabular}}
\end{{table}}"""


print(asn_info_type_to_latex(asn_info_type, date="December, 2025"))


In [ ]:
_stats.write("## Coverage by Network Type (RIS ∩ recent ∩ non-zero)\n\n")
total_asn_types = sum(asn_info_type.values())
for network_type, count in asn_info_type.items():
    pct = count / total_asn_types * 100
    _stats.write(f"- {network_type}: {count:,} ({pct:.1f}%)\n")
_stats.write(f"- Total: {total_asn_types:,}\n")
_stats.write("\n")

### Coverage by topological role (contextualizing the 32%)

Not all ASes play the peering game (reviewer 38D). We split the RIS-visible ASes by CAIDA topological
role and report PeeringDB coverage within each: transit providers (who most need coordinated limits)
are far better covered than single-homed stubs, so the low aggregate reflects the many stubs, not a gap
where limits matter.

In [ ]:
# Contextualize the aggregate coverage (~32% of RIS-visible ASes are in PeeringDB): coverage
# depends strongly on an AS's topological ROLE (CAIDA asnDegree). Networks that provide transit --
# the ones whose peers actually configure max-prefix limits -- are far better covered than the many
# single-homed stubs that neither peer nor need a declared limit. Answers reviewer 38D ("only 32%
# is misleading, not all ASes play the peering game"). No plot; numbers go to the stats file.
as_degree = {}
with open(f"{data_raw_dir}/AS_rank/asns.jsonl") as fd:
    for line in fd:
        d = json.loads(line)
        try:
            asn = int(d["asn"])
        except (ValueError, TypeError):
            continue
        deg = d.get("asnDegree") or {}
        as_degree[asn] = (deg.get("customer", 0) or 0, deg.get("peer", 0) or 0, deg.get("provider", 0) or 0)


def topological_role(asn):
    """Transit = provides transit (has customers); Peering-edge = no customers but peers or
    multi-homed (>=2 upstreams); Stub = single-homed leaf."""
    if asn not in as_degree:
        return "Unknown"
    n_cust, n_peer, n_prov = as_degree[asn]
    if n_cust > 0:
        return "Transit"
    if n_peer > 0 or n_prov >= 2:
        return "Peering-edge"
    return "Stub"


role_groups = {}
for asn in asns_ris:
    role = topological_role(asn)
    role_groups.setdefault(role, {"n": 0, "pdb": 0})
    role_groups[role]["n"] += 1
    if asn in asns_peeringdb:
        role_groups[role]["pdb"] += 1

n_ris = len(asns_ris)
n_ris_pdb = len(asns_ris & asns_peeringdb)

_stats.write("## PeeringDB Coverage by Topological Role (CAIDA asnDegree)\n\n")
_stats.write("Transit = provides transit (has customers); Peering-edge = no customers but peers or "
             "multi-homed (>=2 upstreams); Stub = single-homed leaf.\n\n")
_stats.write(f"Aggregate coverage: {n_ris_pdb:,}/{n_ris:,} = {n_ris_pdb / n_ris * 100:.1f}%\n\n")
_stats.write("| Role | ASes | % of RIS | in PeeringDB | Coverage |\n")
_stats.write("|------|------|----------|--------------|----------|\n")
for role in ["Transit", "Peering-edge", "Stub", "Unknown"]:
    if role not in role_groups:
        continue
    n = role_groups[role]["n"]
    p = role_groups[role]["pdb"]
    _stats.write(f"| {role} | {n:,} | {n / n_ris * 100:.1f}% | {p:,} | {p / n * 100:.1f}% |\n")
    print(f"{role:13} {n:>7,} ({n / n_ris * 100:4.1f}% of RIS)  coverage {p / n * 100:.1f}%")

rel_n = sum(role_groups[r]["n"] for r in ["Transit", "Peering-edge"] if r in role_groups)
rel_p = sum(role_groups[r]["pdb"] for r in ["Transit", "Peering-edge"] if r in role_groups)
_stats.write(f"\n- Peering-relevant (Transit + Peering-edge): {rel_p:,}/{rel_n:,} = "
             f"{rel_p / rel_n * 100:.1f}% (vs {n_ris_pdb / n_ris * 100:.1f}% aggregate)\n\n")
_stats.flush()
print(f"\nTransit coverage {role_groups['Transit']['pdb'] / role_groups['Transit']['n'] * 100:.1f}% | "
      f"Stub {role_groups['Stub']['pdb'] / role_groups['Stub']['n'] * 100:.1f}% | "
      f"peering-relevant {rel_p / rel_n * 100:.1f}%")

## Maximum prefix limit distribution

In [ ]:
df_ris_peeringdb_recent = df_peeringdb[
    df_peeringdb["asn"].isin(asns_ris_peeringdb_recent_nonzero)
]

# maximum prefix limit from RIS & PeeringDB (recently updated ASNs)

# IPv4
df_ris_peeringdb_recent_ipv4 = df_ris_peeringdb_recent[
    df_ris_peeringdb_recent["limits_ipv4"].notna()
].copy()
df_ris_peeringdb_recent_ipv4["last_max_ipv4"] = df_ris_peeringdb_recent_ipv4[
    "limits_ipv4"
].apply(lambda x: x[-1])

ipv4_limits = df_ris_peeringdb_recent_ipv4["last_max_ipv4"].dropna().astype(int)

# IPv6
df_ris_peeringdb_recent_ipv6 = df_ris_peeringdb_recent[
    df_ris_peeringdb_recent["limits_ipv6"].notna()
].copy()
df_ris_peeringdb_recent_ipv6["last_max_ipv6"] = df_ris_peeringdb_recent_ipv6[
    "limits_ipv6"
].apply(lambda x: x[-1])
ipv6_limits = df_ris_peeringdb_recent_ipv6["last_max_ipv6"].dropna().astype(int)


In [ ]:
log_bins = np.linspace(0, 6, 31)
bins = 10**log_bins

for ipv in [4, 6]:

    ipv_limits = ipv4_limits if ipv == 4 else ipv6_limits
    ipv_limits = ipv_limits[ipv_limits > 0]

    counts, _ = np.histogram(ipv_limits, bins=bins)

    fig, ax1 = plt.subplots(figsize=(10, 8))
    bar_width = np.diff(log_bins)[0] * 0.9

    ax1.bar(
        log_bins[:-1],
        counts,
        width=bar_width,
        align="center",
        alpha=0.7,
        color=ipv_color[ipv],
    )

    # set the Ticks and Labels to mimic a Log Scale
    tick_positions = np.arange(0, 7)  # 0, 1, 2, 3, 4, 5, 6
    tick_labels = [f"$10^{{{int(pos)}}}$" for pos in tick_positions]

    ax1.xaxis.set_major_locator(FixedLocator(tick_positions))
    ax1.xaxis.set_major_formatter(FixedFormatter(tick_labels))

    ax1.set_ylabel("Number of ASes", fontsize=two_sided_font_size)
    ax1.set_xlabel("Maximum-Prefix Limit", fontsize=two_sided_font_size)
    ax1.grid(axis="x", linewidth=0.5, alpha=0.75)
    ax1.set_xlim(log_bins[0] - 0.1, log_bins[-1])

    # eCDF on a twin right axis (grey line); keeps the original ePDF bars
    ax2 = ax1.twinx()
    ecdf = np.cumsum(counts) / counts.sum()
    ax2.plot(log_bins[:-1], ecdf, color="grey", lw=2.5)
    ax2.set_ylim(0, 1)
    ax2.set_yticks(np.arange(0.2, 1.01, 0.2))
    ax2.set_ylabel("eCDF", fontsize=two_sided_font_size)

    plt.savefig(
        f"{image_dir}/peering_db/maximum_prefix_distribution_ipv{ipv}.pdf",
        bbox_inches="tight",
        dpi=300,
    )
    plt.savefig(
        f"{image_dir}/peering_db/maximum_prefix_distribution_ipv{ipv}.png",
        bbox_inches="tight",
        dpi=300,
    )
    plt.show()


In [ ]:
plt.figure(figsize=(4, 0.6))
plt.axis("off")
plt.fill_between([], [], [], color="tab:blue", label="IPv4")
plt.fill_between([], [], [], color="tab:green", label="IPv6")
plt.legend(loc="upper right", ncol=2, frameon=False)
plt.savefig(f"{image_dir}/peering_db/legend.pdf", bbox_inches="tight", dpi=300)
plt.savefig(f"{image_dir}/peering_db/legend.png", bbox_inches="tight", dpi=300)
plt.show()


### Mean Evolution of PeeringDB limits

In [ ]:
dates = []
mean_ipv4 = []
mean_ipv6 = []

median_ipv4 = []
median_ipv6 = []

std_ipv4 = []
std_ipv6 = []

for date in historical_df:
    df = historical_df[date]
    ipv4_limits = df["info_prefixes4"]
    ipv4_limits = np.array(ipv4_limits)
    ipv4_limits = ipv4_limits[~np.isnan(ipv4_limits)]

    ipv6_limits = df["info_prefixes6"]
    ipv6_limits = np.array(ipv6_limits)
    ipv6_limits = ipv6_limits[~np.isnan(ipv6_limits)]

    dates.append(date)
    mean_ipv4.append(np.mean(ipv4_limits))
    mean_ipv6.append(np.mean(ipv6_limits))
    std_ipv4.append(np.std(ipv4_limits))
    std_ipv6.append(np.std(ipv6_limits))
    median_ipv4.append(np.median(ipv4_limits))
    median_ipv6.append(np.median(ipv6_limits))


In [ ]:
plt.figure(figsize=(10, 6))


plt.plot(dates, mean_ipv4, marker="o", lw=3, color="tab:blue", alpha=0.8, label="IPv4")
plt.plot(dates, mean_ipv6, marker="s", lw=3, color="tab:green", alpha=0.8, label="IPv6")


plt.ylabel("Mean Maximum Prefix")

plt.xticks(rotation=45)
plt.grid(axis="both", linestyle="--", linewidth=0.5, alpha=0.75)
plt.xlim(dates[0], dates[-1])
plt.ylim(bottom=0)

plt.legend(loc="upper right", bbox_to_anchor=(1, 1.15), ncol=3, frameon=False)

plt.savefig(
    f"{image_dir}/peering_db/temporal_mean_maximum_prefix.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.savefig(
    f"{image_dir}/peering_db/temporal_mean_maximum_prefix.png",
    bbox_inches="tight",
    dpi=300,
)
plt.show()


In [ ]:
plt.figure(figsize=(12, 6))


plt.plot(
    dates, median_ipv4, marker="o", lw=3, color="tab:blue", alpha=0.8, label="IPv4"
)
plt.plot(
    dates, median_ipv6, marker="s", lw=3, color="tab:green", alpha=0.8, label="IPv6"
)


plt.yticks(range(0, 11, 2)) 
plt.ylabel("Median Maximum Prefix")

plt.xticks(rotation=45)
plt.grid(axis="both", linestyle="-", alpha=0.4)
plt.xlim(dates[0], dates[-1])
plt.ylim(bottom=0)

plt.legend(loc="upper right", bbox_to_anchor=(1, 1.2), ncol=3, frameon=False)

plt.savefig(
    f"{image_dir}/peering_db/temporal_median_maximum_prefix.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.savefig(
    f"{image_dir}/peering_db/temporal_median_maximum_prefix.png",
    bbox_inches="tight",
    dpi=300,
)
plt.show()


In [ ]:
print(median_ipv4)
print(median_ipv6)


In [ ]:
_stats.write("## Temporal Median Maximum-Prefix Limits (6-month snapshots, no filter)\n\n")
for date, med4, med6 in zip(selected_dates, median_ipv4, median_ipv6):
    _stats.write(f"- {date.strftime('%Y-%m-%d')}: IPv4={med4:.0f}, IPv6={med6:.0f}\n")
_stats.write("\n")

### Temporal evolution of PeeringDB updates

In [ ]:
df = historical_df[datetime.datetime(2025, 1, 1)]
df["updated"].max()


In [ ]:
selected_date = datetime.datetime(2025, 1, 1)
df_peeringdb_selected = historical_df[selected_date]

df_count = df_peeringdb_selected["updated"].value_counts().sort_index().reset_index()
df_count


In [ ]:
selected_date = datetime.datetime(2025, 1, 1)
df_peeringdb_selected = historical_df[selected_date].copy()

# df_peeringdb_selected = df_peeringdb_selected.apply(
#     lambda row: row if row["updated"] <= selected_date else selected_date, axis=1
# )

df_count = df_peeringdb_selected["updated"].value_counts().sort_index().reset_index()

dates = list(df_count["updated"])
counts = df_count["count"]

dates = np.array(dates)
counts = np.array(counts)

pdf = counts / np.sum(counts)
cdf = np.cumsum(pdf)

cdf = 1 - cdf
months_since_last_update = selected_date - dates
month_days = 30.5
months_since_last_update = [
    (months.days / month_days) for months in months_since_last_update
]

cutoff = selected_date - year_recent_datetime
cutoff_str = year_recent_datetime.strftime("%Y-%m-%d")
months_since_cutoff = cutoff.days / month_days


In [ ]:
plt.figure(figsize=(12, 6))

plt.axvspan(
    0,
    months_since_cutoff,
    color="tab:green",
    alpha=0.1,
)

plt.axvspan(
    months_since_cutoff,
    max(months_since_last_update),
    color="tab:red",
    alpha=0.1,
)

plt.plot(
    months_since_last_update,
    cdf,
    color="tab:grey",
    alpha=0.85,
    lw=3,
    label=selected_date.strftime("%Y-%m-%d"),
)


plt.axvline(
    x=months_since_cutoff,
    color="tab:red",
    alpha=0.7,
    linestyle="--",
)

plt.text(
    14,
    0.7,
    f"Cutoff: {cutoff_str}",
    alpha=0.85,
    fontsize=20,
    bbox=dict(
        facecolor="white",
        boxstyle="round,pad=0.25",
        edgecolor="black",
        alpha=0.7,
        pad=5,
    ),
)

plt.ylabel("Fraction of PeeringDB entries")
plt.xlabel("Months since last update")
plt.ylim(0, 1)

xticks = list(range(0, round(max(months_since_last_update)), 6))
plt.xticks(xticks)
plt.xlim(0, max(months_since_last_update) + 0.1)


plt.grid(axis="both", linestyle="--", linewidth=0.5, alpha=0.75)

plt.savefig(
    f"{image_dir}/peering_db/peeringdb_prefix_limit_updates_cdf.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.savefig(
    f"{image_dir}/peering_db/peeringdb_prefix_limit_updates_cdf.png",
    bbox_inches="tight",
    dpi=300,
)
plt.show()


In [ ]:
bulks_events_updates = df_peeringdb["updated"].apply(lambda updated: datetime.datetime.strftime(updated, "%Y-%m-%d")).value_counts().sort_values(ascending=False)
bulks_events_updates = bulks_events_updates.head(2)
bulks_events_updates.to_dict()

_stats.write("## PeeringDB bulk update dates\n\n")
_stats.write("- PeeringDB bulk update dates (top 2):\n")
for date, count in bulks_events_updates.items():

    percent = count / len(df_peeringdb) * 100

    _stats.write(f"  - {date}: {count:,} entries ({percent:.2f}%)\n")
_stats.write("\n")

## Save selected ASNs

In [ ]:
print(len(asns_ris_peeringdb_recent_nonzero))
fd = open(f"{data_dir}/processed/selected_asns.pkl", "wb")
pickle.dump(asns_ris_peeringdb_recent_nonzero, fd)
fd.close()

In [ ]:
_stats.close()
print(f"Stats written to {numbers_dir}/8-RIPE_PeeringDB_stats.md")
